# SHIPP Stage 9A — Gold listing search documents

`silver_listing_content` → `shipp_gold.gold_listing_search_docs` (the only source of the AI Search index)

**Grain:** one row per **AVAILABLE** listing. **Key:** `listing_id` (traceability back to Lakebase).
**Write mode:** `MERGE` on `doc_hash` — unchanged rows are not touched, removed/unavailable listings are deleted,
Change Data Feed is on, so the Delta Sync index re-embeds only what changed (this is what makes the Velocity path incremental).
No image bytes, no file paths, no coordinates, no PII in this table.

**Clear state and outputs → Run all.** Every gate prints PASS or stops. Never loosen an assert. Only writer of `gold_listing_search_docs`.

In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
# Reload the rag modules so edits to rag/*.py always take effect (same rule as shipp_silver_lib).
for _name in ["rag.text", "rag.vision", "rag.content", "rag.search_docs", "rag.index",
              "rag.retrieval", "rag.semantic", "rag.evaluation"]:
    importlib.reload(importlib.import_module(_name))

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("RAG modules reloaded from", REPO_ROOT)

In [ ]:
from rag import search_docs
from rag.text import pii_rows_filter

TARGET_TABLE = GOLD_LISTING_SEARCH_DOCS
SOURCE_VIEW = "shipp_search_docs_source"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")

docs = search_docs.build_search_docs(spark.table(SILVER_LISTING_CONTENT))
row_count = docs.count()
display(docs.select("listing_id", "title", "status", "vision_status", "search_text").orderBy("listing_id"))

## Step 1 — Quality gate (the index is blocked until this passes)

In [ ]:
run_quality_gate({
    "duplicate listing_id": duplicate_key_count(docs, "listing_id"),
    "null listing_id": null_count(docs, "listing_id"),
    "null search_text": null_count(docs, "search_text"),
    "null doc_hash": null_count(docs, "doc_hash"),
    "status not eligible": docs.filter(~F.col("status").isin(ELIGIBLE_LISTING_STATUSES)).count(),
    "email/phone in search_text": docs.filter(pii_rows_filter(F.col("search_text"))).count(),
}, row_count=row_count, allow_empty=True)

## Step 2 — MERGE into Gold (CDF on)

In [ ]:
spark.sql(search_docs.create_table_sql(TARGET_TABLE))
spark.sql(search_docs.enable_cdf_sql(TARGET_TABLE))
docs.createOrReplaceTempView(SOURCE_VIEW)
merge_metrics = spark.sql(search_docs.merge_sql(TARGET_TABLE, SOURCE_VIEW))
display(merge_metrics)

## Step 3 — Reconcile, schema, CDF, idempotency

In [ ]:
gold = spark.table(TARGET_TABLE)
assert gold.count() == row_count, f"FAILED: Gold has {gold.count()} rows, expected {row_count}"
assert {r.listing_id for r in gold.select("listing_id").collect()} == \
       {r.listing_id for r in docs.select("listing_id").collect()}, "FAILED: Gold keys != source keys"
assert_exact_schema(TARGET_TABLE, search_docs.SEARCH_DOC_COLUMNS)

props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {TARGET_TABLE}").collect()}
assert props.get("delta.enableChangeDataFeed") == "true", "FAILED: Change Data Feed is not enabled"
print("PASS — keys, schema and Change Data Feed.")

rerun = spark.sql(search_docs.merge_sql(TARGET_TABLE, SOURCE_VIEW)).first()
assert rerun["num_affected_rows"] == 0, f"FAILED: second MERGE changed {rerun['num_affected_rows']} rows"
print("PASS — rerun changed 0 rows (idempotent, no needless re-embedding).")

summary = {"table": TARGET_TABLE, "rows": row_count, "status": "PASS"}
print(summary)
dbutils.notebook.exit(json.dumps(summary))